# Anthropic Claude API

The **Anthropic Claude API** is the hosted, closed-weight way to call Claude (Opus, Sonnet, Haiku, and the frontier Fable line). Everything goes through **one endpoint** — `POST /v1/messages` — and the official SDKs (`anthropic` for Python, `@anthropic-ai/sdk` for TS, plus Go/Java/Ruby/C#/PHP) wrap it. Tool use, streaming, vision, structured outputs, prompt caching, and extended thinking are all *features of that one endpoint*, not separate APIs.

**Domain:** Proprietary Models & Coding AI  ·  **recommended addition**  ·  **runnable:** yes  ·  _live calls gated on `ANTHROPIC_API_KEY`_

## 1. What & Why

Anthropic is the lab behind **Claude**. Unlike Llama/Qwen/Mistral's open weights, Claude is **API-only** — you don't download the model; you send a request and pay per token. The API is what you reach for when you want frontier reasoning/coding quality without running any inference infrastructure yourself.

**The problem it solves.** You have a task that needs a strong LLM — classification, extraction, summarization, a chat product, a coding agent — and you don't want to host a 70B+ model, manage GPUs, or wire up your own batching/streaming. You make an HTTPS call; Anthropic runs the model.

**When to reach for it**
- You want **top-tier coding / agentic / long-horizon** quality (this is Claude's strongest suit).
- You need a **stable, supported API** with first-party SDKs, batching, caching, and tool use built in.
- You're building an **agent**: Claude's tool-use loop and 1M-token context are first-class.

**When *not* to**
- You need **open weights** to self-host, fine-tune freely, or run air-gapped → use Llama / Qwen / Mistral.
- The workload is trivial and ultra-high-volume where a tiny local model is cheaper.
- You have a hard **data-residency / zero-egress** rule that forbids sending data to a third party (though Bedrock / Vertex / Claude-on-AWS can soften this).

**One thing that trips people up:** the API is **stateless**. There is no server-side conversation. Every request resends the *entire* history; "memory" is something you build by replaying messages (or via prompt caching to keep it cheap).

## 2. Mental Model

Think of the Claude API as **one function — `messages.create(...)` — that takes a conversation in and returns the next assistant turn out.** Every capability is a parameter or a content-block type on that single call.

```
                       ┌───────────────────────────────────────┐
  model ───────────────▶                                       │
  system (instructions)─▶        POST /v1/messages             │
  messages[] (history) ─▶   (stateless — you send it all)      ├──▶ Message
  tools[] ──────────────▶                                       │     .content[]  (text / thinking / tool_use)
  max_tokens / stream ──▶                                       │     .stop_reason (end_turn | tool_use | max_tokens | refusal)
  thinking / effort ────▶                                       │     .usage       (input / output / cache tokens)
                       └───────────────────────────────────────┘
```

The agent loop is just **calling this function in a `while`**: if `stop_reason == "tool_use"`, run the tool, append the result as a new `user` message, and call again — until `stop_reason == "end_turn"`. That's the whole pattern; the SDK's *tool runner* automates the loop, but it's the same idea underneath.

## 3. Key Concepts

- **Model IDs (use the bare alias).** Current: `claude-opus-4-8` (most capable Opus, 1M ctx), `claude-sonnet-4-6` (speed/intelligence balance, 1M ctx), `claude-haiku-4-5` (fast/cheap, 200K ctx), and `claude-fable-5` (most capable widely-released, frontier). **Never append a date suffix to an alias** (`claude-sonnet-4-6`, *not* `...-20251114`) — a wrong ID 404s. Default to `claude-opus-4-8` unless you have a reason to downgrade.
- **`messages` + `system`.** `messages` is an alternating `user`/`assistant` list (first must be `user`); `system` is a top-level string/array of instructions, **not** a message with `role: "system"`.
- **Content blocks.** Both inputs and outputs are *lists of typed blocks*: `text`, `image`, `document`, `tool_use`, `tool_result`, `thinking`. Always check `block.type` before reading `.text`.
- **`stop_reason`.** Why generation stopped: `end_turn` (done), `tool_use` (run a tool, continue), `max_tokens` (truncated — raise the cap or stream), `refusal` (safety decline — check before reading content), `pause_turn` (server tool paused; resend to resume).
- **`max_tokens`** is the *output* cap and is **required**. Default ~16K for non-streaming, ~64K for streaming. Large values require streaming or the SDK guards against an HTTP timeout.
- **Adaptive thinking & effort.** On Opus 4.6+/Sonnet 4.6/Fable 5 you don't set a thinking *budget*; you use `thinking={"type": "adaptive"}` and tune `output_config={"effort": "low|medium|high|xhigh|max"}`. The old `budget_tokens` 400s on Opus 4.7/4.8/Fable 5.
- **Tool use.** You pass `tools=[{name, description, input_schema}]`; Claude replies with a `tool_use` block; you execute and reply with a `tool_result`. Server-side tools (web search, code execution) run on Anthropic's side.
- **Prompt caching.** Mark a stable prefix with `cache_control` to pay ~0.1× on cache *reads* (up to 90% savings). It's a **prefix match** — any byte change before the breakpoint invalidates it.
- **`usage`.** Every response reports `input_tokens`, `output_tokens`, `cache_creation_input_tokens`, `cache_read_input_tokens` — your cost and cache-hit signal.
- **Batches API.** Async, **50% cheaper**, up to 24h turnaround — for non-latency-sensitive bulk work.

## 4. Setup

```bash
pip install anthropic        # official Python SDK
export ANTHROPIC_API_KEY=sk-ant-...   # get one at https://console.anthropic.com
```

```python
import anthropic
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the env — don't hardcode keys
```

The SDK auto-retries 429/5xx with exponential backoff and exposes typed errors (`anthropic.RateLimitError`, `anthropic.BadRequestError`, …). The cell below checks your environment and runs whether or not the SDK and key are present, so the rest of the notebook executes cleanly either way.

In [ ]:
# Environment probe — runs with or without the SDK / key. No network.
import os

api_key = os.getenv("ANTHROPIC_API_KEY")
try:
    import anthropic  # noqa: F401
    have_sdk = True
    sdk_version = getattr(anthropic, "__version__", "?")
except ImportError:
    have_sdk = False
    sdk_version = None

print("anthropic SDK   :", f"installed (v{sdk_version})" if have_sdk else "(not installed — pip install anthropic)")
print("ANTHROPIC_API_KEY:", "set" if api_key else "(unset — live calls below are skipped)")
print("Live API example will run?", bool(have_sdk and api_key))

## 5. Worked Examples

### Example 1 — Build a Messages request by hand (no network)

The entire API is one JSON body. Assembling it by hand — `system`, alternating `messages`, a tool definition — puts the shape in muscle memory and shows that the SDK is a thin wrapper over this dict. Note `system` is top-level (not a message), the first message is `user`, and a tool is just a name + JSON-Schema `input_schema`.

In [ ]:
import json

# A complete /v1/messages request body — exactly what client.messages.create(**body) sends.
body = {
    "model": "claude-opus-4-8",
    "max_tokens": 1024,                       # required: output cap
    "system": "You are a terse weather assistant.",  # top-level, NOT a message
    "messages": [                              # must start with role 'user' and alternate
        {"role": "user", "content": "What's the weather in Paris?"},
    ],
    "tools": [
        {
            "name": "get_weather",
            "description": "Get current weather for a city. Call this whenever the user asks about weather.",
            "input_schema": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        }
    ],
}

print(json.dumps(body, indent=2))

# Validate the invariants the API enforces:
assert body["messages"][0]["role"] == "user", "first message must be 'user'"
roles = [m["role"] for m in body["messages"]]
assert all(a != b for a, b in zip(roles, roles[1:])), "roles must alternate"
print("\nbody is well-formed:", len(body["messages"]), "message(s),", len(body["tools"]), "tool(s)")

### Example 2 — Estimate token cost before you call (no network)

Pricing is **per million tokens**, split into input and output, with cache reads ~0.1× input and cache writes ~1.25× (5-min TTL). A back-of-envelope estimator catches a "this loop will cost $400" surprise *before* you ship it. (For exact counts use `client.messages.count_tokens(...)` — never `tiktoken`, which is OpenAI's tokenizer and undercounts Claude.)

In [ ]:
# $ per million tokens (input, output). Snapshot — check the pricing page for current numbers.
PRICING = {
    "claude-opus-4-8":   (5.00, 25.00),
    "claude-sonnet-4-6": (3.00, 15.00),
    "claude-haiku-4-5":  (1.00,  5.00),
    "claude-fable-5":    (10.00, 50.00),
}

def estimate_cost(model, input_tokens, output_tokens, cached_input_tokens=0):
    """Rough request cost in USD. Cached input bills at ~0.1x the input rate."""
    in_rate, out_rate = PRICING[model]
    fresh_input = input_tokens - cached_input_tokens
    cost = (fresh_input * in_rate
            + cached_input_tokens * in_rate * 0.1
            + output_tokens * out_rate) / 1_000_000
    return cost

# A chatbot turn: 8K-token system+history prompt, 500-token reply.
for model in PRICING:
    no_cache = estimate_cost(model, 8_000, 500)
    with_cache = estimate_cost(model, 8_000, 500, cached_input_tokens=7_500)  # stable prefix cached
    print(f"{model:20s}  no-cache ${no_cache:.5f}   cached-prefix ${with_cache:.5f}")

savings = 1 - estimate_cost('claude-opus-4-8', 8_000, 500, 7_500) / estimate_cost('claude-opus-4-8', 8_000, 500)
print(f"\nPrompt caching saves ~{savings:.0%} on this Opus turn (the cached prefix bills at 0.1x).")

### Example 3 — A real call + the tool-use loop (gated on `ANTHROPIC_API_KEY`)

The live path. With the SDK installed and `ANTHROPIC_API_KEY` set, this sends a real request and walks the agent loop; otherwise it prints the exact call shape so the notebook still runs top-to-bottom. The loop is the whole pattern: **call → if `stop_reason == "tool_use"`, run the tool and append a `tool_result` → call again → until `end_turn`.**

In [ ]:
# Live tool-use loop, gated so the notebook runs with or without a key / the SDK.
def run_weather(city):
    return f"18C and clear in {city}"  # your real implementation goes here

def agent_turn(client, user_msg):
    tools = body["tools"]  # reuse the get_weather tool defined in Example 1
    messages = [{"role": "user", "content": user_msg}]
    while True:
        resp = client.messages.create(
            model="claude-opus-4-8", max_tokens=1024, tools=tools, messages=messages,
        )
        if resp.stop_reason != "tool_use":
            return next((b.text for b in resp.content if b.type == "text"), "")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                out = run_weather(**b.input)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})

if have_sdk and api_key:
    client = anthropic.Anthropic()
    answer = agent_turn(client, "What's the weather in Paris? Answer in one sentence.")
    print("Claude:", answer)
else:
    print("[skipped — set ANTHROPIC_API_KEY and `pip install anthropic` to run live]")
    print("Call shape that would run:")
    print("  client.messages.create(model='claude-opus-4-8', max_tokens=1024,")
    print("                         tools=tools, messages=messages)")
    print("  -> loop while resp.stop_reason == 'tool_use', appending tool_result blocks")

### Example 4 — Streaming (call shape)

For anything with long output or high `max_tokens`, **stream** — it avoids HTTP timeouts and lets you show tokens as they arrive. The SDK's `messages.stream()` accumulates state for you; `get_final_message()` returns the complete `Message` (with `usage`) at the end.

```python
with client.messages.stream(
    model="claude-opus-4-8",
    max_tokens=64000,
    messages=[{"role": "user", "content": "Write a short poem about caching."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)   # tokens arrive incrementally
    final = stream.get_final_message()     # full Message, including .usage
    print("\n\noutput tokens:", final.usage.output_tokens)
```

## 6. Gotchas & Pitfalls

- **The API is stateless.** There's no server-side conversation — resend the full `messages` history every turn. Forgetting this gives you an amnesiac bot. Use **prompt caching** to keep the resent prefix cheap.
- **`system` is not a message.** It's a top-level parameter. Putting `{"role": "system", ...}` as `messages[0]` 400s (except the beta mid-conversation-system feature, which is *not* the initial prompt).
- **First message must be `user` and roles must alternate.** `assistant` first, or two `user` turns in a row, → 400.
- **Reading `content[0].text` blindly.** Output is a *list of typed blocks* — there may be `thinking` or `tool_use` blocks before/instead of text, and on a `refusal` the content can be empty. Always check `block.type` and `stop_reason` first.
- **`budget_tokens` is dead on the latest models.** Opus 4.7/4.8 and Fable 5 reject `thinking={"type":"enabled","budget_tokens":N}` with a 400. Use `thinking={"type":"adaptive"}` + `output_config={"effort": ...}`. `temperature`/`top_p`/`top_k` are also removed on those models.
- **Dated model IDs.** Don't append a date to an alias from memory (`claude-opus-4-8-20251114`) — it 404s. Use the bare alias.
- **Lowballing `max_tokens`.** Hitting the cap gives `stop_reason == "max_tokens"` and truncates mid-sentence. Raise it (and stream above ~16K).
- **Silent cache misses.** A `datetime.now()`, a UUID, or unsorted `json.dumps` in the cached prefix changes the bytes and invalidates the cache. If `usage.cache_read_input_tokens` is 0 across identical-prefix requests, a silent invalidator is at work.
- **Assistant prefill is gone** on Opus 4.6+/Sonnet 4.6/Fable 5 (the old "put words in Claude's mouth" trick 400s). Use **structured outputs** (`output_config.format`) or a system instruction instead.
- **`tiktoken` lies.** It's OpenAI's tokenizer; it undercounts Claude by ~15–20%. Use `client.messages.count_tokens(...)`.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Best coding / agentic / long-horizon quality** | **Claude API** (`claude-opus-4-8`, `claude-fable-5`) | Claude's strongest domain; native tool-use loop + 1M context. |
| **Speed/cost balance at scale** | `claude-sonnet-4-6` | ~Opus-class for many tasks at lower price. |
| **Cheap, fast, simple tasks** (classify, route) | `claude-haiku-4-5` | Lowest latency and price. |
| **Open weights — self-host, fine-tune, air-gap** | **Llama / Qwen / Mistral** | Claude is API-only; you can't download it. |
| **A different frontier vendor / ecosystem** | **OpenAI GPT / Google Gemini** | Comparable closed APIs; pick by eval, price, and tooling fit. |
| **Bulk, non-latency-sensitive jobs** | **Claude Batches API** | 50% cheaper, async (≤24h). |
| **Stay inside your cloud / compliance boundary** | **Bedrock / Vertex / Claude on AWS** | Same Claude via your cloud's auth & billing (some features differ). |

**Honest trade-offs.** You give up weight access, full fine-tuning, and offline use; cost is per-token (can dominate at high volume); and you're sending data to a third party (mitigable via cloud providers). In return you get frontier quality, zero infra, and batteries-included tool use, caching, batching, and streaming. Cross-link: see the **Mistral**, **LLaMA**, and **Qwen** notebooks for the open-weight side, and **Google Gemini / Vertex** for the other closed-API option.

## 8. Resources

- **API & docs home** — https://docs.claude.com/en/docs/get-started
- **Messages API reference** — https://docs.claude.com/en/api/messages
- **Models overview & pricing** — https://docs.claude.com/en/docs/about-claude/models/overview
- **Tool use guide** — https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview
- **Prompt caching** — https://docs.claude.com/en/docs/build-with-claude/prompt-caching
- **Python SDK (`anthropic`)** — https://github.com/anthropics/anthropic-sdk-python
- **Console (get an API key)** — https://console.anthropic.com

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def agent_loop(call, tools, user_message, run_tool, max_turns=10):
    """Drive the request -> tool -> request cycle until the model says it is done."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE